# MLflow Tracking Server

**MLflow** is an open-source platform that simplifies the tracking, comparison, and deployment of machine learning experiments.

In this sample example template, you’ll use MLflow to **track training runs**, **log parameters and metrics**, and **store models** for future reuse — all within a single notebook.



Install **MLflow**, **Gradio**, and supporting libraries including **scikit‑learn**, **matplotlib**, and **pandas**.


In [ ]:
!pip install -q mlflow scikit-learn matplotlib pandas gradio


Import MLflow, perform a quick GPU check with PyTorch, and load helper libraries used throughout.


In [ ]:
import mlflow, os, torch, pandas as pd, matplotlib.pyplot as plt, gradio as gr
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Using device: {device}')
if device == 'cpu':
    print('⚠️ Running on CPU — switch to GPU for faster performance if available.')


By default, MLflow saves runs to the local **`mlruns/`** directory. You can switch to a **remote tracking server** later by setting a different tracking URI.


In [ ]:
mlflow.set_tracking_uri('file:///content/mlruns')
mlflow.set_experiment('mlflow_tracking_demo')
print('🎯 Tracking URI:', mlflow.get_tracking_uri())


It fetches experiment metadata, parameters, and metrics from your local `mlruns/` directory (or a remote server if configured).


In [ ]:
from mlflow.tracking import MlflowClient

def show_mlflow_runs_table(experiment_name="mlflow_tracking_demo"):
    """Display all MLflow runs (similar to MLflow UI Table)."""
    client = MlflowClient()
    experiment = client.get_experiment_by_name(experiment_name)

    if not experiment:
        return pd.DataFrame({"Info": ["No experiment found. Run a training cell first."]})
    runs = client.search_runs([experiment.experiment_id])
    if not runs:
        return pd.DataFrame({"Info": ["No runs logged yet."]})

    rows = []
    for r in runs:
        row = {
            "Run ID": r.info.run_id,
            "Status": r.info.status,
            "Start Time": pd.to_datetime(r.info.start_time, unit="ms"),
            "End Time": pd.to_datetime(r.info.end_time, unit="ms"),
            "Duration (s)": round((r.info.end_time - r.info.start_time) / 1000, 2)
            if r.info.end_time else None,
        }
        row.update(r.data.params)
        row.update(r.data.metrics)
        rows.append(row)

    df = pd.DataFrame(rows)
    main_cols = ["Run ID", "Status", "Start Time", "End Time", "Duration (s)"]
    other_cols = [c for c in df.columns if c not in main_cols]
    df = df[main_cols + other_cols]
    print(f"✅ Showing {len(df)} runs from experiment '{experiment_name}'")
    return df

runs_df = show_mlflow_runs_table()
display(runs_df)


Let's train a small **Random Forest** on the Diabetes dataset and log parameters, metrics, and the model artefact to MLflow.


In [ ]:
from mlflow.models.signature import infer_signature

with mlflow.start_run() as run:
    db = load_diabetes()
    X_train, X_test, y_train, y_test = train_test_split(db.data, db.target, test_size=0.2, random_state=42)

    model = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    signature = infer_signature(X_test, preds)

    mlflow.log_params({'n_estimators': 100, 'max_depth': 6})
    mlflow.log_metric('mean_prediction', float(preds.mean()))
    mlflow.sklearn.log_model(model, 'model', signature=signature)

    print(f'Run ID: {run.info.run_id}')
    print('✅ Training and logging complete!')


Use the run ID to load the stored model for inference.


In [ ]:
run_id = run.info.run_id
loaded_model = mlflow.sklearn.load_model(f'runs:/{run_id}/model')
print('✅ Model loaded successfully!')
print('Sample predictions:', loaded_model.predict(X_test[:5]))


So, you've configured MLflow tracking locally (can be configure for MLflow remote server too), logged parameters, metrics, and model artifacts.

Additionally, you can reload a trained model from specific run using the `run Id`. Guide on deployment on saturn can be found in the [saturn documentation](https://saturncloud.io/docs).